# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 11_12_15_main_geometry_generation  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-03-03
### Properties script
**Goal:** Main script that generates the geometry dataset that acts as input for the structural analysis in grasshopper, this script can take several variables as input at generates a CSV file made up of properties that can generate this digital geometry to a CAD geometry   
**Inputs:**   
*   Define base mesh
*   Allowed movement for different vertices
*   divisions over allowed movement

**Outputs:**

*   CSV file, with sample id, vertices, coordinates and edges

## 1.1 IMPORTING PARAMETERS

In [ ]:
import pandas as pd
import numpy as np
import c11_params
import config

### Testing

In [ ]:
from geometry import get_corner_indices

# --- TESTEN ---
grid_2x2 = get_corner_indices(2, 2)
print(f"2x2 Grid Indices: {grid_2x2}")

grid_3x3 = get_corner_indices(3, 3)
print(f"3x3 Grid Indices: {grid_3x3}")

grid_4x4 = get_corner_indices(4, 4)
print(f"4x4 Grid Indices: {grid_4x4}")
# Verwacht: 0 (BL), 4 (BR), 20 (TL), 24 (TR) -> want 5 punten per rij

# In je loop:
corners = get_corner_indices(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)
corner_values = corners.values() # [0, 4, 20, 24] bijv.
print(corner_values)

## 1.2 Executing and saving

In [ ]:
from geometry import generate_full_dataset, generate_edges
from config import RAW_DATA_PATH

# --- EXECUTE ---
final_vertices = generate_full_dataset(c11_params.NUM_SAMPLES)
final_edges = generate_edges(c11_params.NUM_SAMPLES, c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)

# --- SAVE ---
final_vertices.to_csv(RAW_DATA_PATH / f"dataset_{c11_params.GRID}_{c11_params.NUM_SAMPLES}_vertices.csv", index=False)
final_edges.to_csv(RAW_DATA_PATH / f"dataset_{c11_params.GRID}_{c11_params.NUM_SAMPLES}_edges.csv", index=False)

print(f"Generatie gereed voor {c11_params.NUM_SAMPLES} samples.")
print(f"Grid grootte: {c11_params.GRID}")

print("Vertices Shape:", final_vertices.shape)
print("Edges Shape:", final_edges.shape)

print(f"\nTotaal aantal regels in vertices CSV: {len(final_vertices)}")
print("\nVoorbeeld output (eerste 5 regels van sample 0):")
print(final_vertices.head(5))
print("\nVoorbeeld output (eerste 5 regels van sample 1):")
print(final_vertices[final_vertices['sample_id'] == 1].head(5))

print(f"\nTotaal aantal regels in edges CSV: {len(final_edges)}")
print("\nVoorbeeld output (eerste 5 regels van sample 0):")
print(final_edges.head(5))
print("\nVoorbeeld output (eerste 5 regels van sample 1):")
print(final_edges[final_edges['sample_id'] == 1].head(5))

## 1.5 DESIGN DOMAIN
two output from the geometry script need to be transferred to json files for further use in the script, these are:
*   Search Space, this is the space where the vertices can move, this is nessecary for the optimizing of the geometry so the optimizer knows where it can and cant go.
*   Edge Index, the edge index is nessecary for the training of the surrogate model, this is used because the relationship of the vertices is the same vor every structure, also lowers this the number of features (x) used in training for a better generalisation.

In [11]:
import json
import config
import c11_params
from data_io import define_search_space
from geometry import generate_edges

# --- UITVOEREN EN BEKIJKEN ---
# Gebruik de variabelen uit je eerdere configuratie
search_space = define_search_space(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y, c11_params.DIVISIONS, c11_params.EDGE_LENGTH)
df_topology = generate_edges(num_samples=1, cells_x=c11_params.GRID_CELLS_X, cells_y=c11_params.GRID_CELLS_Y)
edge_index = df_topology[['V1', 'V2']].values.T.tolist()

print(f"De Search Space bevat in totaal {len(search_space)} onafhankelijke variabelen (knoppen waar de AI aan mag draaien).")
print(f"Topologie gegenereerd. Aantal unieke verbindingen (edges): {len(df_topology)}")
print("\nVoorbeeld van hoe de computer dit ziet:")

# Toon de eerste 5 variabelen mooi geformatteerd
for var_name, bounds in list(search_space.items())[:5]:
    print(f"- {var_name}: {bounds}")

print("\nGegenereerde edge_index (eerste 5 verbindingen):")
print(f"Bron-nodes (V1): {edge_index[0][:5]}")
print(f"Doel-nodes (V2): {edge_index[1][:5]}")

De Search Space bevat in totaal 18 onafhankelijke variabelen (knoppen waar de AI aan mag draaien).
Topologie gegenereerd. Aantal unieke verbindingen (edges): 32

Voorbeeld van hoe de computer dit ziet:
- v1_shift_x: {'type': 'discrete', 'options': [-1.125, -0.75, -0.375, 0.0, 0.375, 0.75, 1.125]}
- v3_shift_y: {'type': 'discrete', 'options': [-1.125, -0.75, -0.375, 0.0, 0.375, 0.75, 1.125]}
- v4_shift_x: {'type': 'discrete', 'options': [-1.125, -0.75, -0.375, 0.0, 0.375, 0.75, 1.125]}
- v4_shift_y: {'type': 'discrete', 'options': [-1.125, -0.75, -0.375, 0.0, 0.375, 0.75, 1.125]}
- v5_shift_y: {'type': 'discrete', 'options': [-1.125, -0.75, -0.375, 0.0, 0.375, 0.75, 1.125]}

Gegenereerde edge_index (eerste 5 verbindingen):
Bron-nodes (V1): [0, 0, 1, 1, 2]
Doel-nodes (V2): [1, 3, 2, 4, 5]


In [12]:
# Sla de dictionarys op als een JSON bestand
json_search_space_path = 'search_space.json'
json_edge_index_path = 'edge_index.json'
with open(json_search_space_path, 'w') as f:
    json.dump(search_space, f, indent=4) # indent=4 maakt het mooi leesbaar
with open(json_edge_index_path, 'w') as f:
    json.dump(edge_index, f)

print(f"Search Space succesvol opgeslagen als '{json_search_space_path}'")
print(f"\nEdge index (als pure lijst) succesvol opgeslagen in '{json_edge_index_path}'.")

Search Space succesvol opgeslagen als 'search_space.json'

Edge index (als pure lijst) succesvol opgeslagen in 'edge_index.json'.
